<a href="https://colab.research.google.com/github/AnaraHayat/flyrank_assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #4 — The Freshness Multiplier** (growth-to-decline ratio by freshness window).

The paper reports a 361+ day freshness bucket at a 283:1 growth-to-decline ratio, and to its credit flags this itself: *"the `361+` bucket... is too small and too unstable to treat as a headline multiplier"* because there is only 1 declining page in that bucket. That's the right instinct — a ratio built on a single-digit denominator can swing from 283:1 to 30:1 by moving one row. My question is narrower: the paper doesn't state whether the growth/decline counts behind the *stable* 31-90d ratio (7.88:1) are pooled across all 57 brands or reported per-brand and averaged. If one or two large brands dominate the pooled count, the ratio describes those brands' content calendars more than a portfolio-wide pattern. Worth a sentence in the methodology section either way.

**ML Appendix — Growth Prediction (Logistic Regression, 71% holdout accuracy).**

The label is `trend_direction` from a 30-day-vs-previous-30-day impression comparison — the same kind of target I use for `is_declining_label`. The Methodology page states the split is an *"80/20 split"* for the Random Forest, Logistic Regression, and Decision Tree — with no mention of grouping by brand. The portfolio spans 57 brands with very uneven page counts per brand (mirroring my own 32-client dataset, from 3 to 7,008 rows per client). If that 80/20 split is a random **row** split rather than a **brand-holdout** split, the same brand's pages can land in both train and test, and the 71% figure would partly measure "did the model memorize this brand's content calendar" rather than "does this generalize to a brand it hasn't seen." Section 2 below measures how large that gap can be, using my own data as a stand-in since I can't re-run the paper's exact split without their raw tables.

In [2]:
import os

REPO_URL = "https://github.com/AnaraHayat/flyrank_assignment1.git"
REPO_DIR = "/content/flyrank_assignment1"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
from pathlib import Path

DATA_REL = "data/raw/content_refresh_anonymized.csv"
start = Path.cwd()
repo_root = None
for candidate in [start, *start.parents]:
    if (candidate / DATA_REL).exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError(f"Couldn't find {DATA_REL} above {start}. Run the git-clone cell first if this is Colab.")
os.chdir(repo_root)

df = pd.read_csv(DATA_REL)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print(f"Rows: {len(df)}, clients: {df['client_id'].nunique()}")

Cloning into '/content/flyrank_assignment1'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 145 (delta 52), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1.89 MiB | 10.41 MiB/s, done.
Resolving deltas: 100% (52/52), done.
Rows: 30000, clients: 32


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = ["competition_level", "content_type", "main_intent"]
missing_prone = ["search_volume", "competition", "cpc", "word_count", "char_count"]
for c in missing_prone:
    df[f"has_{c}"] = df[c].notna().astype(int)
missing_flag_features = [f"has_{c}" for c in missing_prone]

X = df[numeric_features + categorical_features + missing_flag_features].copy()
for c in numeric_features + missing_flag_features:
    X[c] = X[c].fillna(0)
for c in categorical_features:
    X[c] = X[c].fillna("unknown")

groups = df["client_id"]
prep = ColumnTransformer([
    ("num", StandardScaler(), numeric_features + missing_flag_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

def run(train_idx, test_idx, label):
    lr = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))])
    lr.fit(X.iloc[train_idx], y[train_idx])
    scores = lr.predict_proba(X.iloc[test_idx])[:, 1]
    y_test = y[test_idx]
    p20, p50, p100 = (precision_at_k(scores, y_test, k) for k in (20, 50, 100))
    n_test_clients = groups.iloc[test_idx].nunique()
    print(f"{label}: test_clients={n_test_clients:<3} P@20={p20:.3f}  P@50={p50:.3f}  P@100={p100:.3f}")
    return p50

# GROUPED split -- same client-holdout design as w05_model.ipynb
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_g, test_idx_g = next(gss.split(X, y, groups=groups))
p50_grouped = run(train_idx_g, test_idx_g, "GROUPED (client-holdout, honest)      ")

# RANDOM ROW split -- same clients can appear in both train and test, like an ungrouped 80/20
train_idx_r, test_idx_r = train_test_split(np.arange(len(df)), test_size=0.25, random_state=42, stratify=y)
p50_random = run(train_idx_r, test_idx_r, "RANDOM ROW SPLIT (ungrouped, like the paper's stated 80/20)")

print()
print(f"Gap: the ungrouped split reads P@50 {p50_random - p50_grouped:+.3f} higher than the honest grouped split.")

GROUPED (client-holdout, honest)      : test_clients=8   P@20=0.800  P@50=0.780  P@100=0.730
RANDOM ROW SPLIT (ungrouped, like the paper's stated 80/20): test_clients=31  P@20=1.000  P@50=0.940  P@100=0.850

Gap: the ungrouped split reads P@50 +0.160 higher than the honest grouped split.


**Before/after, in one line:** the same Logistic Regression, same data, same K — grouped client-holdout scores P@50 = 0.78; an ungrouped random-row split (31 of 32 clients leaking across train and test) scores P@50 = 0.94. That's a 0.16 gap from split design alone, with nothing else changed. This isn't a claim that the paper's 71% figure is wrong — I don't have their raw split to check — it's a demonstration of *how large* the gap an ungrouped split can hide, using a dataset with the same brand/client imbalance shape as theirs. It's the concrete argument behind the methodology question in Section 1: worth one sentence in the paper's methodology page confirming whether the 80/20 splits were grouped by brand.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# Re-run the Week-3 leakage hunt (ML-04 / ML-05) against the FINAL feature set actually used in w05/w06,
# not just the draft list from Week 3.

final_features = set(numeric_features) | set(categorical_features) | set(missing_flag_features)
label_source_cols = {"trend_direction", "trend_pct"}
post_label_window_cols = {
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
identifier_cols = {"content_id", "client_id"}
derived_tier_cols = {
    "age_tier", "age_tier_order", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier",
}  # binned copies of numeric features already included directly -- not leakage, just redundant

print("Label-source columns present in final features?", final_features & label_source_cols)
print("30d-window columns (built from the same window as the label) present in final features?", final_features & post_label_window_cols)
print("Identifier columns present in final features?", final_features & identifier_cols)
print("Redundant tier columns present in final features?", final_features & derived_tier_cols)
print()
print("Result: final feature set is clean -- 0 label-source columns, 0 post-label-window columns,")
print("0 identifier columns, 0 redundant tier columns. Matches the Week-3 data contract (ML-04).")

# Second check: does any single feature correlate suspiciously highly (>0.9) with the label?
# A near-perfect correlate is the classic tell that a column secretly encodes the label.
numeric_only = df[numeric_features].fillna(0)
corrs = numeric_only.corrwith(pd.Series(y, index=df.index)).abs().sort_values(ascending=False)
print()
print("Top 5 |correlation| with is_declining_label (looking for anything suspiciously close to 1.0):")
print(corrs.head(5).round(3))

Label-source columns present in final features? set()
30d-window columns (built from the same window as the label) present in final features? set()
Identifier columns present in final features? set()
Redundant tier columns present in final features? set()

Result: final feature set is clean -- 0 label-source columns, 0 post-label-window columns,
0 identifier columns, 0 redundant tier columns. Matches the Week-3 data contract (ML-04).

Top 5 |correlation| with is_declining_label (looking for anything suspiciously close to 1.0):
days_with_impressions     0.190
content_age_days          0.164
word_count                0.119
char_count                0.108
days_since_last_update    0.081
dtype: float64


No leakage found. The final feature set has zero overlap with the label-source columns (`trend_direction`, `trend_pct`), zero overlap with the 30-day-window columns the label is computed from, no identifiers, and no redundant tier duplicates. The strongest single-feature correlation with the label is well under 0.9 — no column is secretly standing in for the answer. This matches the Week-3 data contract and gives the P@50 numbers in Section 2 (and in w05) a clean floor to stand on.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
boldest_claim = (
    "Logistic Regression clearly outperforms both the Week-4 baseline rule and Random Forest at P@50."
)
safe_rewrite = (
    "On this snapshot, under a client-holdout split, Logistic Regression's P@50 (0.78) was observed to be "
    "higher than both the Week-4 baseline rule (0.56) and the Random Forest variant tested (0.66). This is a "
    "single held-out sample of 8 clients, so the gap is directional evidence for prioritizing Logistic "
    "Regression in a decision-support role -- flagging pages for human review -- not a guarantee it will "
    "rank first on a different client mix or a different 90-day window."
)
print("BOLD:", boldest_claim)
print()
print("SAFE:", safe_rewrite)

BOLD: Logistic Regression clearly outperforms both the Week-4 baseline rule and Random Forest at P@50.

SAFE: On this snapshot, under a client-holdout split, Logistic Regression's P@50 (0.78) was observed to be higher than both the Week-4 baseline rule (0.56) and the Random Forest variant tested (0.66). This is a single held-out sample of 8 clients, so the gap is directional evidence for prioritizing Logistic Regression in a decision-support role -- flagging pages for human review -- not a guarantee it will rank first on a different client mix or a different 90-day window.


The bold version reads like a settled fact. The safe version keeps the same number but is honest about what it rests on: one client-holdout split, 8 test clients, a single 90-day snapshot, and a scoring model — not a deployed, causal, or repeated-trial result. "Observed" and "directional" do real work here: they tell a reader this is evidence worth acting on, not a promise about every future client.

## Self-check

Before you submit, confirm each line honestly:

- [Done ] Every section above is filled — markdown thinking AND the code that backs it
- [ Done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done ] No client names, URLs, or private queries anywhere
- [DOne ] My claims use careful words: observed, measured, directional, decision-support
- [Done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.